##### AIT 726 - Final Project Naive Bayes Code
##### By Japneet Kohli

####  The code is executed in the following order.

1.   Load data.
2.   Select only translation writer and line translations data from the list of attributes.
3.   Assign labels of translation writer attribute binary numeric values (0,1).
4.   Preprocess text using regex.
5.   Split data into train, validation, and test.
6.   Create pipeline with functions including Multinomial Naive Bayes, Count Vectorizer, and TF-IDF Transformer, and assign range of hyerparameters to be tuned.
7.   Perform 10 fold cross validation using GridSearchCV to find best values for hyperparameters. 
8.   Fit the model with the tuned parameters on training data.
9.   Predict values for validation and test data and compute accuracy using classfication report and confusion matrix.
10.  Save the validation prediction results for offline analysis.

In [ ]:
# import packages
import os
import re
import numpy as np
import pandas as pd
from nltk import word_tokenize
from nltk.stem.porter import PorterStemmer
import logging

# use logging to save the results
logging.basicConfig(filename='baselinemodel.log', level=logging.INFO)
logging.getLogger().addHandler(logging.StreamHandler())
print('Done')

In [ ]:
# assign stemmer
stemmer = PorterStemmer()
print('Done')

In [ ]:
# read data
directory = r'csv_output_files'
dfpath = directory + "\\" + 'datadf.csv'
data=pd.read_csv(dfpath)
print('Done')

In [ ]:
# display data
data.head()

In [ ]:
# keep only listed columns in dataframe
columns_to_keep = ['writer','lines_translations_english_writer','lines_translations_english','lines_type']
newdata=data[columns_to_keep]

In [ ]:
# display data
newdata.head()

In [ ]:
# describe data
newdata.describe()

In [ ]:
# display data info
newdata.info()

In [ ]:
# display writer column by grouping count values
newdata.writer.value_counts()

In [ ]:
# display translator data by grouping count values
newdata.lines_translations_english_writer.value_counts()

In [ ]:
# keep data for only listed columns. here, its the label (translator writer) and text (translations) 
translator_columns=['lines_translations_english_writer','lines_translations_english']
translator_data=newdata[translator_columns]

In [ ]:
# display data
translator_data.head()

In [ ]:
# display data shape
translator_data.shape

In [ ]:
# describe data
translator_data.describe()

In [ ]:
# display data info
translator_data.info()

In [ ]:
# display data at random row
translator_data.lines_translations_english[14]

In [ ]:
# display data for translations
translator_data["lines_translations_english"]

In [ ]:
# display values for translator writer
translator_data.lines_translations_english_writer.values

In [ ]:
# replace translator writer label with binary 0 and 1
translator_data["lines_translations_english_writer"].replace({"Dr. Sant Singh Khalsa": "0", "Bhai Manmohan Singh": "1"}, inplace=True)

In [ ]:
# display data to see if binary values were correctly placed
translator_data.lines_translations_english_writer.values

In [ ]:
# display value counts for label
translator_data.lines_translations_english_writer.value_counts()

In [ ]:
# preprocess data to remove punctuation, html tags, non ascii characters, etc.
def preprocess_text(txt):
        txt = re.sub(r'(?:<[^>]+>)', '', txt)#Html tags
        txt = re.sub('[^\x00-\x7F]+', ' ', txt) ##remove non ascii charcters
        txt = re.sub('#\S+', '', txt)
        txt = re.sub('_', '', txt)
        txt = re.sub('-',' ',txt)
        txt = re.sub('/','',txt)
        txt = re.sub('~','',txt)
        txt = re.sub('|','',txt)
        txt = re.sub('\s+', ' ', txt) ## remove double spaces
        txt = re.sub(r'([A-Z][a-z]+)',lambda x:x.group().lower(),txt)##Lower case captilized words only but not words with all capitals
        return txt.strip()
data = [preprocess_text(t) for t in translator_data["lines_translations_english"]]
labels= translator_data.lines_translations_english_writer.values
print('Done')

In [ ]:
data

In [ ]:
labels

In [ ]:
# import required packages for NB and split data into train, test, validation
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import GaussianNB
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.model_selection import train_test_split, GridSearchCV
x_train, x_test, y_train, y_test = train_test_split(data, labels, test_size=0.20, random_state=42)
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.20, random_state=42)
print('Done')

In [ ]:
x_train

In [ ]:
y_train

In [ ]:
# assign pipeine with functions and hyperparameters for turing
text_clf = Pipeline([('vect', CountVectorizer()),
                     ('tfidf', TfidfTransformer()),
                     #('to_dense', DenseTransformer()),
                     ('clf', MultinomialNB())])

tuned_parameters = {
    'vect__ngram_range': [(1, 1), (1, 2), (2, 2)],
    'tfidf__use_idf': (True, False),
    'tfidf__norm': ('l1', 'l2'),
    'clf__alpha': [1e-1, 1e-2, 1e-3]
}
print('Done')

In [ ]:
df_y_train= pd.DataFrame(y_train, columns=['y_train'])

In [ ]:
df_y_train.y_train.value_counts()

In [ ]:
len(x_val)

In [ ]:
len(x_test)

In [ ]:
# tune hyperparameters and get best value using 10-fold cross validation
# fit model on training data and print mean, standard deviation values for hyperparameters tuned
from sklearn.metrics import classification_report

score = 'f1_micro'
print("# Tuning hyper-parameters for %s" % score)
print()
np.errstate(divide='ignore')
clf = GridSearchCV(text_clf, tuned_parameters, cv=10, scoring=score)
clf.fit(x_train, y_train)

print("Best parameters set found on development set:")
print()
print(clf.best_params_)
print()
print("Grid scores on development set:")
print()
for mean, std, params in zip(clf.cv_results_['mean_test_score'], 
                             clf.cv_results_['std_test_score'], 
                             clf.cv_results_['params']):
    print("%0.3f (+/-%0.03f) for %r" % (mean, std * 2, params))
print()

print("Detailed classification report:")
print()
print("The model is trained on the full development set.")

In [ ]:
# predict training data
y_train_pred = clf.predict(x_train)

In [ ]:
# classification report for training data
print(classification_report(y_train, y_train_pred, digits=4))

In [ ]:
# predict validation data
y_val_pred= clf.predict(x_val)

In [ ]:
# classification report for validation data
print(classification_report(y_val, y_val_pred, digits=4))

In [ ]:
# confusion matrix for validation data
from sklearn import metrics
print(metrics.confusion_matrix(y_val, y_val_pred))

In [ ]:
# predict test data
y_test_pred = clf.predict(x_test)

In [ ]:
# classification report for test data
print(classification_report(y_test, y_test_pred, digits=4))

In [ ]:
# confusion matrix for test data 
print(metrics.confusion_matrix(y_test, y_test_pred))

In [ ]:
# create dataframe of validation data
df_y_val_true=pd.DataFrame(y_val, columns=['y_true']) 
df_y_val_true

In [ ]:
# add validation predictions in df
df_y_val_true['y_pred']=y_val_pred
df_y_val_true

In [ ]:
# rename
df_val=df_y_val_true

In [ ]:
# add text column (translations)
df_val['lines_translations_english'] = x_val

In [ ]:
# display data
df_val.head(5)

In [ ]:
# export data
df_val.to_excel(r'validation_nb.xlsx')